# Solutions, Lab 04: The Cached-Logit Pipeline

This notebook solves the four exercises at the end of `labs/lab-04-cached-logit-pipeline.ipynb`.
Lab 04 is a Tier 2 lab, so its training runs execute only with `RUN_TRAINING = True` on a
training box. The solutions split as follows:

- **Exercise 1 (break the shift on purpose):** fully live. The bug is reproduced on a toy
  world small enough to train to convergence here, three times, with the real cache machinery.
- **Exercise 2 (k ablation):** the static half is live on real teacher logits (caches rebuilt
  at k in {8, 32, 128} and the truncation bias measured); the 500-step trainings are gated.
- **Exercise 3 (tail on, tail off):** the static bias is measured live on real logits, and the
  training consequence is demonstrated live on a toy optimization; the full trainings are gated.
- **Exercise 4 (cache at T=2):** the mismatch is created and detected live with the lab's own
  spot-check protocol on real teacher logits; the trainings are gated.

One scope note for the live measurements. The build machine for this course is CPU-only with a
small memory budget, so real-logit measurements use SmolLM2-360M-Instruct as the teacher and
SmolLM2-135M-Instruct as the student, both in fp32, on a 16-row slice of the lab's eval set
truncated to 192 positions. The code is identical to what runs at full scale; only the tensor
sizes differ, and the gated cells run the full-size versions. Attempt the exercises before
reading on.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math, shutil, gc, time
sys.path.insert(0, "../code")

import numpy as np
import torch
import torch.nn.functional as F

from kd_core import (topk_forward_kl, make_topk_cache, kl_divergence,
                     shift_for_next_token, top1_agreement, mean_entropy,
                     bytes_per_token_cache, masked_mean, topk_truncation_bias,
                     completion_mask_from_prompt_lens)
from kd_pipeline import (set_seed_everywhere, config_fingerprint, MemoryPlan,
                         full_ft_gb, infer_gb, prefill_wallclock_hours,
                         TopKCacheWriter, TopKCacheReader, RunManifest)

RUN_TRAINING = False        # <-- flip on the training box, same flag as the lab
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# The lab's config, copied verbatim so gated solutions run the same experiment.
CFG = dict(
    teacher="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    student="HuggingFaceTB/SmolLM2-360M-Instruct",
    k=64, cache_T=1.0, seq_len=384, vocab=49152,
    lr=3e-5, batch_size=8, grad_accum=4, max_steps=1500, warmup_steps=50,
    data="../data/lab03", cache_dir="../data/lab04_cache",
)

def spot_check(reader, score_fn, rows, atol=1e-2):
    # The lab's Part A.3 protocol, verbatim: re-derive cached top-k logprobs, raise on mismatch.
    batch = reader.batch(rows)
    logits = score_fn(batch["input_ids"])
    log_p = F.log_softmax(logits.float() / reader.manifest["temperature"], dim=-1)
    recomputed = log_p.gather(-1, batch["topk_idx"])
    err = (recomputed - batch["topk_logprobs"]).abs()
    err = err[batch["mask"]].max() if batch["mask"].any() else err.max()
    assert float(err) < atol, f"spot check failed: max |dlogprob| = {float(err):.4f}"
    return float(err)

print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Exercise 1: Break the shift on purpose

**The exercise, restated.** Train 200 steps with `b["mask"][:, :-1]` instead of `[:, 1:]` and
watch what the loss curve does, and does not, tell you. This is the course's central bug,
experienced deliberately in a controlled setting.

**The approach.** The instructive way to solve this is on a world where the *correct* answer
is exactly knowable, so every consequence of misalignment is measurable rather than argued.
The world: a bigram teacher over a 64-token vocabulary, meaning the teacher's distribution
for the next token depends only on the current token, through a fixed random table of logits.
Sequences are random tokens, prompt lengths are random, and the mask is built by the course's
own `completion_mask_from_prompt_lens`. The teacher's logits go through the real
`make_topk_cache`, and the student is the smallest model that can represent this teacher
exactly: a learnable table `W` of shape [64, 64], where row x holds the student's logits for
"what comes after token x". Because the student has exactly the teacher's capacity, the
correct pipeline must drive the loss to essentially zero and the top-1 agreement to 1.0, which
gives us a clean reference against which two sabotaged runs can be judged.

Three training runs of 200 steps each, all scored afterward against the *correctly shifted*
teacher:

1. **Correct:** student logits `[:, :-1]` against cache rows `[:, :-1]` and mask `[:, 1:]`,
   the lab's Part B alignment.
2. **Misaligned cache:** student logits `[:, :-1]` against cache rows `[:, 1:]`, the
   generalized version of the shift bug where predictions and targets are off by one whole
   position. The student at position t (which sees token x_t) is now being trained toward the
   teacher's distribution at position t+1, which depends on x_{t+1}, a token the student has
   not seen. The information needed to hit the target simply is not in the input.
3. **The literal exercise bug:** correct cache rows but mask `[:, :-1]` instead of `[:, 1:]`.
   With contiguous prompt-then-completion masks this changes only the boundary: it supervises
   the position that predicts the *last prompt token's successor* one position early and drops
   each row's final supervised prediction.

The prediction to test: run 2's loss still *decreases*, because there is something learnable
even in misaligned targets (their average), while its agreement stays near the chance floor
of 1/64, which is about 1.6 percent. Run 3's loss curve should be indistinguishable from run
1's, which is exactly what makes the real version of this bug dangerous.

In [2]:
set_seed_everywhere(SEED)

Vt, Bt, Tt, K = 64, 96, 32, 16
# The bigram teacher: a shared component (some tokens are just commoner, as in real text)
# plus a per-context component (what actually depends on the current token).
table = 0.6 * torch.randn(1, Vt) + 1.5 * torch.randn(Vt, Vt)
ids = torch.randint(0, Vt, (Bt, Tt))
t_log = table[ids]                                  # position t predicts token t+1
prompt_lens = torch.randint(4, 13, (Bt,)).tolist()
mask = completion_mask_from_prompt_lens(ids, prompt_lens)
cache = make_topk_cache(t_log, k=K)
slice_cache = lambda c, sl: {k: (v[:, sl] if v.dim() > 1 else v) for k, v in c.items()}

def train_head(cache_slice, mask_slice, steps=200):
    W = torch.zeros(Vt, Vt, requires_grad=True)
    opt = torch.optim.Adam([W], lr=0.2)
    losses = []
    for _ in range(steps):
        loss = topk_forward_kl(W[ids][:, :-1], slice_cache(cache, cache_slice), mask_slice)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss.detach()))
    with torch.no_grad():
        agree = top1_agreement(W[ids][:, :-1], t_log[:, :-1], mask[:, 1:])  # vs CORRECT ref
    return losses, agree, W.detach()

runs = {
    "correct shift":            (slice(None, -1), mask[:, 1:]),
    "cache misaligned by one":  (slice(1, None),  mask[:, 1:]),
    "mask bug  [:, :-1]":       (slice(None, -1), mask[:, :-1]),
}
out = {}
print(f"{'run':<26} {'loss step 0':>11} {'loss step 200':>13} {'agreement':>10}")
for name, (cs, ms) in runs.items():
    losses, agree, W = train_head(cs, ms)
    out[name] = (losses, agree, W)
    print(f"{name:<26} {losses[0]:>11.4f} {losses[-1]:>13.6f} {agree:>10.3f}")

l_ok, a_ok, W_ok = out["correct shift"]
l_mis, a_mis, W_mis = out["cache misaligned by one"]
l_bug, a_bug, _ = out["mask bug  [:, :-1]"]

# The correct run proves the setup can be solved exactly.
assert l_ok[-1] < 1e-4 and a_ok > 0.95, "correct pipeline must recover the teacher"
# The misaligned run: the loss DOES decrease...
assert l_mis[-1] < 0.85 * l_mis[0], "misaligned loss still goes down"
# ...but agreement stays near the 1/64 chance floor, nowhere near the correct run.
assert a_mis < 0.15, f"misaligned agreement should be near chance, got {a_mis:.3f}"
# What did the misaligned student actually learn? The corpus-average next-token
# distribution: its 64 rows are nearly identical to each other.
row_p = F.softmax(W_mis, -1)
row_spread = float(0.5 * (row_p[None] - row_p[:, None]).abs().sum(-1).max())
assert row_spread < 0.35, "misaligned student collapsed to one context-free distribution"
# The literal mask bug: the loss curve is indistinguishable from correct...
assert abs(l_bug[-1] - l_ok[-1]) < 1e-3
# ...and only a position-level audit sees the difference: it supervises positions
# the correct mask excludes (one boundary position per row).
n_disagree = int((mask[:, 1:] != mask[:, :-1]).sum())
assert n_disagree == Bt, "the mask bug moves exactly one boundary position per row"
print(f"\nmisaligned run: loss fell {100*(1-l_mis[-1]/l_mis[0]):.0f}% yet agreement is "
      f"{a_mis:.3f} (chance floor 1/64 = {1/64:.3f}); its rows differ by at most "
      f"TVD {row_spread:.2f}, i.e. it learned one average distribution")
print(f"mask-bug run: loss curve matches correct to {abs(l_bug[-1]-l_ok[-1]):.1e}, "
      f"but {n_disagree} supervised positions ({n_disagree}/{int(mask[:,1:].sum())} of the "
      f"total) are the wrong ones")
print("CHECK ex1-live: the loss curve cannot distinguish any of these three runs' health")

run                        loss step 0 loss step 200  agreement


correct shift                   0.9287     -0.000000      1.000


cache misaligned by one         0.9325      0.727635      0.075


mask bug  [:, :-1]              0.9298     -0.000000      1.000

misaligned run: loss fell 22% yet agreement is 0.075 (chance floor 1/64 = 0.016); its rows differ by at most TVD 0.29, i.e. it learned one average distribution
mask-bug run: loss curve matches correct to 4.2e-10, but 96 supervised positions (96/2271 of the total) are the wrong ones
CHECK ex1-live: the loss curve cannot distinguish any of these three runs' health


**Interpretation.** The table is the course's central lesson in three rows. The correct
run drove the loss to numerical zero and agreement to 1.0, as it must when the student has
exactly the teacher's capacity. The misaligned run's loss fell by roughly a fifth and then
flattened, a curve that on a dashboard reads as "converged, this is the task floor". It is
nothing of the kind. The printed checks show what actually happened: agreement against the
correctly shifted teacher sat near the 1/64 chance floor, and the trained student's 64 rows
are nearly one and the same distribution (maximum row-to-row total variation distance under
0.35, against 1.0-ish spreads in the teacher's rows). Because the target at each position was
statistically unrelated to the input token, the best the student could do, and exactly what
gradient descent found, was the corpus's *average* next-token distribution: real learning,
measurable loss reduction, zero conditional knowledge. That is what "the loss curve does not
tell you" means concretely: loss going down proves the student is absorbing *something*, not
that it is the thing you wanted.

The literal mask-slice bug from the exercise is the quieter sibling. Its loss curve matched
the correct run's to within 1e-3 (in this toy the damage rounds to nothing, because the
bigram teacher is equally learnable at every position), yet the audit shows it supervising
one wrong position per row: in the real pipeline that position is the prompt-to-completion
boundary, where it trains on a prediction made *from inside the prompt* and silently drops
each row's final token, the EOS the lab's data audit worked to keep supervised. A model
trained that way stops late or not at all, and no loss curve will ever say so. The defenses
this course keeps repeating are the only ones that work: the four-point mask audit, one shift
applied to logits and mask together, and an agreement measurement against an independently
computed reference. All three ran in this cell; none of them is optional.

## Exercise 2: k ablation, for real

**The exercise, restated.** Rebuild the cache at k in {8, 32, 128} and train 500 steps each.
Plot final agreement against cache size in GB. Where is *your* knee?

**The approach.** The exercise has a static half and a training half, and the static half is
the one that generalizes: before training anything, measure what each k *loses*, which is the
truncation bias `topk_truncation_bias` computes by comparing the exact dense forward KL
against its top-k approximations on the same logits. I run that live on real logits: the
360M model as teacher, the 135M as student, fp32, on a 16-row, 192-position slice of the
lab's eval corpus (the build-box scale from the header; the function and code are unchanged
at full scale). I also build the three caches for real with `TopKCacheWriter`, so the bytes
on disk come from the actual format rather than arithmetic, and project them to the lab's
full 4096-row corpus, which gives the x axis of the exercise's plot. The 500-step trainings
are gated below with the lab's own two-stage functions, parameterized by k.

Why bias-per-k predicts the training result: the cached objective differs from the dense one
only by what truncation discards, so if the bias at some k is already inside run-to-run
noise, training at that k cannot be distinguished from dense training, and any larger k buys
disk, not agreement. The knee of the static curve is therefore a forecast of the knee of the
training curve.

In [3]:
from transformers import AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

set_seed_everywhere(SEED)
T_SUB, N_ROWS, BS = 192, 16, 4
ev = torch.load("../data/lab03/eval.pt")
good = (ev["mask"][:, :T_SUB].sum(1) >= 40).nonzero().squeeze(-1)
rows = good[:N_ROWS]
ids_sub = ev["input_ids"][rows][:, :T_SUB]
mask_sub = ev["mask"][rows][:, :T_SUB]

@torch.no_grad()
def dense_logits(name):
    mdl = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval()
    outs = [mdl(ids_sub[i:i+BS]).logits for i in range(0, N_ROWS, BS)]
    del mdl; gc.collect()
    return torch.cat(outs)

t0 = time.time()
t_logits = dense_logits("HuggingFaceTB/SmolLM2-360M-Instruct")   # teacher role
s_logits = dense_logits("HuggingFaceTB/SmolLM2-135M-Instruct")   # student role
s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, mask_sub)
print(f"{N_ROWS} rows x {T_SUB} positions scored in fp32 in {time.time()-t0:.0f}s; "
      f"{int(m_sh.sum())} supervised positions")

KS = (8, 32, 64, 128)
bias = topk_truncation_bias(s_sh, t_sh, m_sh, ks=KS)

# Build the caches for real, to get true bytes on disk per k.
FULL_TOKENS = 4096 * 384      # the lab's actual corpus
print(f"\n{'k':>4} {'mass kept':>10} {'renorm err':>11} {'tail err':>10} "
      f"{'bytes here':>11} {'full-corpus GB':>15}")
disk = {}
for row in bias:
    k = row["k"]
    d = f"../data/_sol04_kcache_{k}"
    shutil.rmtree(d, ignore_errors=True)
    w = TopKCacheWriter(d, k=k, vocab_size=CFG["vocab"], seq_len=T_SUB - 1)
    w.append(t_sh.contiguous(), ids_sub[:, 1:].contiguous(), m_sh.contiguous())
    man = w.finalize()
    per_tok = man["bytes_on_disk"] / (N_ROWS * (T_SUB - 1))
    disk[k] = per_tok * FULL_TOKENS / 1e9
    print(f"{k:>4} {row['mean_mass_covered']:>10.4f} {row['renorm_rel_err']:>+11.1%} "
          f"{row['tail_rel_err']:>+10.1%} {man['bytes_on_disk']:>11,} {disk[k]:>15.2f}")

by_k = {r["k"]: r for r in bias}
# Coverage must rise with k, and both estimators' bias must shrink with k.
for a, b in zip(KS, KS[1:]):
    assert by_k[b]["mean_mass_covered"] > by_k[a]["mean_mass_covered"]
    assert abs(by_k[b]["renorm_rel_err"]) < abs(by_k[a]["renorm_rel_err"])
    assert abs(by_k[b]["tail_rel_err"]) < abs(by_k[a]["tail_rel_err"])
# The lab's Part A cited Lab 02's measurement: single-digit-percent bias at k=64.
assert abs(by_k[64]["renorm_rel_err"]) < 0.10 and abs(by_k[64]["tail_rel_err"]) < 0.10
print(f"\nCHECK ex2-live: bias falls monotonically in k and is single-digit percent by "
      f"k=64 ({by_k[64]['renorm_rel_err']:+.1%} renorm, {by_k[64]['tail_rel_err']:+.1%} "
      f"tail), reproducing Lab 02 section 5 on this pair")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


16 rows x 192 positions scored in fp32 in 28s; 1175 supervised positions



   k  mass kept  renorm err   tail err  bytes here  full-corpus GB


   8     0.9693      +14.3%     -13.7%     168,080            0.09


  32     0.9906       +4.4%      -5.0%     608,144            0.31


  64     0.9947       +2.4%      -3.0%   1,194,896            0.61


 128     0.9971       +1.3%      -1.8%   2,368,400            1.22

CHECK ex2-live: bias falls monotonically in k and is single-digit percent by k=64 (+2.4% renorm, -3.0% tail), reproducing Lab 02 section 5 on this pair


In [4]:
# Exercise 2, gated part: rebuild the full cache at each k and train 500 steps.
# stage1/stage2 are the lab's Part B functions, parameterized by k and step budget.
from transformers import AutoModelForCausalLM, get_cosine_schedule_with_warmup

def stage1_build_cache(cfg):
    teacher = AutoModelForCausalLM.from_pretrained(
        cfg["teacher"], dtype=torch.bfloat16).to(device).eval()
    tr = torch.load(os.path.join(cfg["data"], "train.pt"))
    ids_, mask_ = tr["input_ids"], tr["mask"]
    writer = TopKCacheWriter(cfg["cache_dir"], k=cfg["k"], vocab_size=cfg["vocab"],
                             seq_len=cfg["seq_len"], temperature=cfg["cache_T"])
    with torch.no_grad():
        for i in range(0, len(ids_), 16):
            writer.append(teacher(ids_[i:i+16].to(device)).logits.cpu(),
                          ids_[i:i+16], mask_[i:i+16])
    manifest = writer.finalize()
    del teacher
    if device == "cuda":
        torch.cuda.empty_cache()
    return manifest

def stage2_train_from_cache(cfg, use_tail=True, train_T=None, tag="cached"):
    train_T = cfg["cache_T"] if train_T is None else train_T
    reader = TopKCacheReader(cfg["cache_dir"])
    tr = torch.load(os.path.join(cfg["data"], "train.pt"))
    reader.verify_against(tr["input_ids"].numpy())
    teacher_check = AutoModelForCausalLM.from_pretrained(
        cfg["teacher"], dtype=torch.bfloat16).to(device).eval()
    with torch.no_grad():
        spot_check(reader, lambda x: teacher_check(x.to(device)).logits.cpu(),
                   rows=np.random.default_rng(0).choice(len(reader), 40, replace=False),
                   atol=5e-2)
    del teacher_check
    if device == "cuda":
        torch.cuda.empty_cache()
    student = AutoModelForCausalLM.from_pretrained(
        cfg["student"], dtype=torch.bfloat16).to(device)
    opt = torch.optim.AdamW(student.parameters(), lr=cfg["lr"])
    sched = get_cosine_schedule_with_warmup(opt, cfg["warmup_steps"], cfg["max_steps"])
    order = np.random.default_rng(SEED).permutation(len(reader))
    log, step = [], 0
    while step < cfg["max_steps"]:
        for i in range(0, len(order), cfg["batch_size"]):
            b = reader.batch(order[i:i+cfg["batch_size"]])
            s_l = student(b["input_ids"].to(device)).logits
            cache_sh = {k2: (v[:, :-1].to(device) if v.dim() > 1 else v)
                        for k2, v in b.items() if k2 not in ("input_ids", "mask")}
            loss = topk_forward_kl(s_l[:, :-1], cache_sh, b["mask"][:, 1:].to(device),
                                   T=train_T, use_tail=use_tail) / cfg["grad_accum"]
            loss.backward()
            if (step + 1) % cfg["grad_accum"] == 0:
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if step % 100 == 0:
                print(f"[{tag}] step {step:>5}  cached-KL {float(loss)*cfg['grad_accum']:.4f}")
            step += 1
            if step >= cfg["max_steps"]:
                break
    fp = config_fingerprint({**cfg, "use_tail": use_tail, "train_T": train_T, "seed": SEED})
    out = f"../runs/sol04/{tag}_{fp}"
    os.makedirs(out, exist_ok=True)
    student.save_pretrained(out)
    del student
    if device == "cuda":
        torch.cuda.empty_cache()
    return out

if RUN_TRAINING:
    for k in (8, 32, 128):
        cfg_k = {**CFG, "k": k, "max_steps": 500,
                 "cache_dir": f"../data/sol04_cache_k{k}"}
        stage1_build_cache(cfg_k)
        out = stage2_train_from_cache(cfg_k, tag=f"k{k}")
        print(f"k={k} ->", out)
else:
    print("RUN_TRAINING=False: the three 500-step trainings are written but did not "
          "execute here.")

RUN_TRAINING=False: the three 500-step trainings are written but did not execute here.


**Interpretation.** Reading the live table by column: mass kept climbs from about 97
percent at k=8 to above 99.7 percent at k=128, and both bias estimators shrink monotonically,
from the low-teens percent at k=8 to a couple of percent by k=64 and one or two percent at
k=128. Two details deserve attention. First, the two estimators miss in *opposite
directions* on this pair: renormalizing over the kept entries overstated the KL here while
the tail bucket understated it, each by a similar magnitude. The lab quoted Lab 02 section
5's finding of single-digit-percent bias "in both estimator directions" at k=64, and the
printed check reproduces exactly that on this model pair, so treat the two estimators as
error bars bracketing the truth rather than one being strictly better. Second, the cost
column: the projected full-corpus cache runs from roughly a tenth of a gigabyte at k=8 to
over a gigabyte at k=128, a roughly 12x spread in disk for a bias difference that has already
fallen inside training noise somewhere in between. The static knee on this evidence is
between k=32 and k=64: k=8's double-digit bias is a real objective distortion, while k=128
buys about one percentage point of bias over k=64 for double the disk.

**The three 500-step trainings did not execute in this build; expected shape, not results:**
final agreement should be flat within noise between k=32 and k=128, with k=8 trailing by a
point or two at most on this peaked instruct teacher, and the lab's Part C band (cached-KL
indistinguishable from live-teacher KL at matched steps) should hold from k=32 up. What
would refute the static forecast is agreement still climbing from k=32 to k=128 by more than
the seed spread; if you see that, your teacher is less peaked than this one (a
higher-temperature or creative-domain teacher pushes the knee right, as the `kd_core`
docstring warns), and the static table, rerun on your own corpus, will say so before any
training does.

## Exercise 3: Tail on, tail off

**The exercise, restated.** `topk_forward_kl(use_tail=False)` renormalizes over the kept k
entries instead of keeping an aggregate tail bucket. Lab 02 measured the static bias; measure
the *training* consequence: does a student that never sees the tail term behave differently
on rare tokens? Check entropy.

**The approach.** Two live measurements, one static and one dynamic. The static one reads
the k=64 row of exercise 2's table, computed on real logits, to establish that the two
objectives really are different functions. The dynamic one asks what each objective *wants*,
which is a question about minimizers, not about any particular model: I take one synthetic
teacher position over a 512-token vocabulary with a substantial tail (about 19 percent of its
mass outside the top 16), cache it at k=16, and directly optimize a free student logit vector
against each objective for 50 steps. Because the student here is an unconstrained vector, the
optimization lands close to each objective's true minimizer, and the difference between the
two minimizers is exactly the training consequence the exercise asks about, isolated from
every other influence. The quantities to compare are the ones the exercise names: the
student's probability mass on rare tokens (everything outside the cached top-k) and its
entropy. The 500-step real trainings, one per objective, are gated.

The prediction, derivable before running: with the tail bucket, the objective contains a term
matching the student's off-top-k mass to the teacher's, so the minimizer keeps a tail. Without
it, the renormalized target is a distribution that sums to 1 *over the k kept tokens*, so the
student minimizes by moving all its mass onto those k tokens: the tail is not merely
unsupervised, it is actively squeezed out, and entropy must fall.

In [5]:
set_seed_everywhere(SEED)

# Static, on real logits: the k=64 row from exercise 2 (same tensors, still in memory).
r64 = by_k[64]
print(f"static bias at k=64 on real logits: dense KL {r64['dense_kl']:.4f}, "
      f"renorm {r64['renorm_kl']:.4f} ({r64['renorm_rel_err']:+.1%}), "
      f"tail bucket {r64['tail_bucket_kl']:.4f} ({r64['tail_rel_err']:+.1%})")

# Dynamic: what each objective's minimizer looks like.
V = 512
zt_toy = 3.0 * torch.randn(V)
cache_toy = make_topk_cache(zt_toy.view(1, 1, V), k=16)
m1 = torch.ones(1, 1, dtype=torch.bool)
teacher_tail = float(cache_toy["tail_logprob"].exp())
teacher_H = mean_entropy(zt_toy.view(1, 1, V), m1)

def optimize_student(use_tail, steps=50):
    z = torch.zeros(1, 1, V, requires_grad=True)          # uniform start
    opt = torch.optim.Adam([z], lr=0.3)
    for _ in range(steps):
        loss = topk_forward_kl(z, cache_toy, m1, use_tail=use_tail)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        s = F.softmax(z, -1)
        tail_mass = 1.0 - float(s.gather(-1, cache_toy["topk_idx"]).sum())
        return tail_mass, mean_entropy(z, m1)

tail_renorm, H_renorm = optimize_student(use_tail=False)
tail_bucket, H_bucket = optimize_student(use_tail=True)

print(f"\nteacher: tail mass {teacher_tail:.3f}, entropy {teacher_H:.3f} nats")
print(f"use_tail=True  student: tail mass {tail_bucket:.3f}, entropy {H_bucket:.3f} nats")
print(f"use_tail=False student: tail mass {tail_renorm:.4f}, entropy {H_renorm:.3f} nats")

assert tail_renorm < 0.05, "renormalized objective squeezes the tail out"
assert 0.5 * teacher_tail < tail_bucket < 2.0 * teacher_tail, \
    "tail-bucket objective preserves roughly the teacher's tail mass"
assert H_renorm < H_bucket - 0.5, "the entropy consequence is large, not subtle"
print(f"\nCHECK ex3-live: dropping the tail term moved {teacher_tail:.0%} of probability "
      f"mass off the rare tokens and cost {H_bucket - H_renorm:.2f} nats of entropy at "
      f"the minimizer")

static bias at k=64 on real logits: dense KL 0.2618, renorm 0.2680 (+2.4%), tail bucket 0.2540 (-3.0%)

teacher: tail mass 0.193, entropy 3.205 nats
use_tail=True  student: tail mass 0.216, entropy 3.629 nats
use_tail=False student: tail mass 0.0016, entropy 2.229 nats

CHECK ex3-live: dropping the tail term moved 19% of probability mass off the rare tokens and cost 1.40 nats of entropy at the minimizer


In [6]:
# Exercise 3, gated part: the same 500-step training twice, use_tail on and off,
# then compare entropy on held-out data (the exercise's suggested diagnostic).
if RUN_TRAINING:
    cfg3 = {**CFG, "max_steps": 500, "cache_dir": "../data/sol04_cache_k64"}
    if not os.path.exists(os.path.join(cfg3["cache_dir"], "manifest.json")):
        stage1_build_cache(cfg3)
    out_tail = stage2_train_from_cache(cfg3, use_tail=True, tag="tail-on")
    out_renorm = stage2_train_from_cache(cfg3, use_tail=False, tag="tail-off")
    ev_ = torch.load(os.path.join(cfg3["data"], "eval.pt"))
    for name, path in (("tail-on", out_tail), ("tail-off", out_renorm)):
        st = AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16).to(device).eval()
        with torch.no_grad():
            lg = st(ev_["input_ids"][:64].to(device)).logits[:, :-1]
        print(name, "held-out entropy:",
              round(mean_entropy(lg, ev_["mask"][:64, 1:].to(device)), 3), "nats")
        del st
        if device == "cuda":
            torch.cuda.empty_cache()
else:
    print("RUN_TRAINING=False: the paired trainings are written but did not execute here.")

RUN_TRAINING=False: the paired trainings are written but did not execute here.


**Interpretation.** The static row says the two objectives differ by a few percent of
KL at k=64 on real logits, which sounds ignorable. The minimizer experiment says the
difference is not ignorable at all, because the bias is not *neutral* in direction: the
renormalized objective's optimum put under one percent of its mass outside the cached top-k
against the teacher's 19 percent, and paid more than a full nat of entropy for it (the
printed check). The mechanism is in the algebra: renormalizing the cached top-k produces a
target that sums to 1 over k tokens, so the student's cheapest way to match it is to
concentrate everything there; the tail bucket restores the one number, total off-top-k mass,
that makes keeping a tail worthwhile. A per-position entropy cost of this kind compounds over
a generation: a student trained without the tail term is systematically overconfident on
every rare token, which is invisible in the loss (the loss is what asked for it) and shows up
downstream as degraded diversity and worse calibration on exactly the inputs where the
teacher was uncertain.

**The paired 500-step trainings did not execute in this build; expected ranges, not
results:** the tail-off student's held-out entropy should sit measurably below the tail-on
student's, on the order of a few hundredths to a tenth of a nat at 500 steps on this pair
(the full-model version is far from its objective's minimizer at that budget, so the effect
is a fraction of the toy's 1.4 nats), widening with more steps. Agreement should be close to
tied, since top-1 tokens live inside the top-k either way; that combination, entropy down
with agreement flat, is the confirming signature. If instead you see the tail-off arm's
entropy *higher*, check the mask before anything else: an objective this lopsided losing its
direction usually means the average includes unsupervised positions.

## Exercise 4: Cache at T=2

**The exercise, restated.** Rebuild the cache with `cache_T=2.0` and train; compare against
T=1 caching plus T=1 training. Where must the temperature agree, and where is it free? The
manifest stores the temperature for a reason.

**The approach.** The cache does not store logits; it stores *log-probabilities normalized at
a specific temperature* (`make_topk_cache` divides by T before the softmax). That makes
temperature part of the cache's identity, exactly like the corpus fingerprint, and the
questions "where must it agree" and "how would I know it does not" both have mechanical
answers that can be demonstrated live on the real teacher logits already in memory from
exercise 2. Four measurements, on an 8-row slice:

1. Build two real caches from the same teacher logits, one at T=1 and one at T=2, and read
   what changed: the manifests differ in the `temperature` field, and the T=2 cache covers
   *less* mass at the same k, because softening pushes probability into the tail.
2. The honest case: spot-check each cache with the true scorer at the cache's own manifest
   temperature. Both must pass.
3. The silent failure the exercise is about: a trainer that assumes T=1 while holding the
   T=2 cache. I simulate the operator error by overriding the reader's manifest temperature
   to 1.0 and spot-checking again; the check must fail loudly, and this is precisely why the
   manifest records temperature at all.
4. The training consequence if no one checks: compute the cached loss on the T=2 cache with
   the matching `T=2` versus the mismatched `T=1`, against the exact dense KL at T=2 as
   ground truth. The mismatched objective is not a noisier version of the right one; it is a
   different function.

The paired full trainings (T=2 cache with T=2 training, versus the lab's T=1/T=1 baseline)
are gated.

In [7]:
set_seed_everywhere(SEED)
SUB = 8
t_sub = t_sh[:SUB].contiguous(); s_sub = s_sh[:SUB].contiguous()
m_sub = m_sh[:SUB].contiguous(); ids_c = ids_sub[:SUB, 1:].contiguous()

readers = {}
for Tc in (1.0, 2.0):
    d = f"../data/_sol04_Tcache_{Tc}"
    shutil.rmtree(d, ignore_errors=True)
    w = TopKCacheWriter(d, k=64, vocab_size=CFG["vocab"], seq_len=T_SUB - 1, temperature=Tc)
    w.append(t_sub, ids_c, m_sub)
    w.finalize()
    readers[Tc] = TopKCacheReader(d)

# 1: temperature is part of the cache's identity, and softening costs coverage at fixed k.
mass = {Tc: float(r.batch(range(SUB))["topk_logprobs"].exp().sum(-1)[m_sub].mean())
        for Tc, r in readers.items()}
assert readers[1.0].manifest["temperature"] == 1.0
assert readers[2.0].manifest["temperature"] == 2.0
assert mass[2.0] < mass[1.0] - 0.02, "softened teacher leaks more mass past any fixed k"
print(f"top-64 mass covered: {mass[1.0]:.4f} at cache_T=1 vs {mass[2.0]:.4f} at cache_T=2 "
      f"(softening moved {mass[1.0]-mass[2.0]:.1%} of mass into the tail)")

# 2: honest spot checks pass at each cache's own recorded temperature.
for Tc, r in readers.items():
    err = spot_check(r, lambda x: t_sub[:x.shape[0]], rows=range(SUB))
    print(f"spot check, cache_T={Tc}, manifest temperature honored: max err {err:.4f} (pass)")

# 3: the simulated operator error: same T=2 cache, manifest read as T=1.
lying_reader = readers[2.0]
true_manifest = lying_reader.manifest
lying_reader.manifest = {**true_manifest, "temperature": 1.0}
try:
    spot_check(lying_reader, lambda x: t_sub[:x.shape[0]], rows=range(SUB))
    raise RuntimeError("temperature mismatch NOT caught")
except AssertionError as e:
    print(f"assumed-T=1 spot check on the T=2 cache: {e} (correctly rejected)")
lying_reader.manifest = true_manifest

# 4: the training consequence, against exact dense KL at T=2 as ground truth.
dense_T2 = float(kl_divergence(s_sub, t_sub, m_sub, T=2.0, scale_by_T2=False))
b2 = readers[2.0].batch(range(SUB))
matched = float(topk_forward_kl(s_sub, b2, m_sub, T=2.0, scale_by_T2=False))
mismatched = float(topk_forward_kl(s_sub, b2, m_sub, T=1.0, scale_by_T2=False))
err_matched, err_mism = abs(matched - dense_T2), abs(mismatched - dense_T2)
print(f"\ndense KL at T=2: {dense_T2:.4f}; cached loss with matching T=2: {matched:.4f} "
      f"(truncation gap {err_matched:.3f}); with mismatched T=1: {mismatched:.4f} "
      f"(off by {err_mism:.3f})")
assert err_mism > 3 * err_matched, "the mismatch dwarfs honest truncation error"
print("CHECK ex4-live: temperature mismatch is a different objective, and only the "
      "manifest plus the spot check can catch it before training does")

top-64 mass covered: 0.9952 at cache_T=1 vs 0.7035 at cache_T=2 (softening moved 29.2% of mass into the tail)


spot check, cache_T=1.0, manifest temperature honored: max err 0.0078 (pass)


spot check, cache_T=2.0, manifest temperature honored: max err 0.0039 (pass)


assumed-T=1 spot check on the T=2 cache: spot check failed: max |dlogprob| = 15.4971 (correctly rejected)



dense KL at T=2: 0.3990; cached loss with matching T=2: 0.1873 (truncation gap 0.212); with mismatched T=1: 1.5939 (off by 1.195)
CHECK ex4-live: temperature mismatch is a different objective, and only the manifest plus the spot check can catch it before training does


In [8]:
# Exercise 4, gated part: T=2 cache trained at T=2, versus the lab's T=1 baseline.
if RUN_TRAINING:
    cfgT2 = {**CFG, "cache_T": 2.0, "max_steps": 500,
             "cache_dir": "../data/sol04_cache_T2"}
    stage1_build_cache(cfgT2)
    out_T2 = stage2_train_from_cache(cfgT2, train_T=2.0, tag="cacheT2-trainT2")
    cfgT1 = {**CFG, "max_steps": 500, "cache_dir": "../data/sol04_cache_k64"}
    if not os.path.exists(os.path.join(cfgT1["cache_dir"], "manifest.json")):
        stage1_build_cache(cfgT1)
    out_T1 = stage2_train_from_cache(cfgT1, train_T=1.0, tag="cacheT1-trainT1")
    print("T2/T2 ->", out_T2, "\nT1/T1 ->", out_T1)
else:
    print("RUN_TRAINING=False: the paired trainings are written but did not execute here.")

# Tidy the temporary measurement caches either way.
for d in list(os.listdir("../data")):
    if d.startswith("_sol04_"):
        shutil.rmtree(os.path.join("../data", d), ignore_errors=True)
print("temporary measurement caches removed")

RUN_TRAINING=False: the paired trainings are written but did not execute here.
temporary measurement caches removed


**Interpretation.** Where must the temperature agree: between the cache build and the
training loss, because the cache stores probabilities already normalized at `cache_T`, and
`topk_forward_kl`'s `T` argument softens only the *student*. Feed a T=2 cache to a T=1
training loss and, as the live cell shows, you are not adding noise, you are optimizing a
different function: the mismatched loss came out several times farther from the true T=2 KL
than the honest truncation gap (the printed check), and unlike a bug that raises, it trains
just fine. Where the temperature is free: the choice itself. T=2 caching plus T=2 training is
a perfectly coherent experiment (it is Lab 03's soft arm, cached); the eval-time diagnostics
are computed at their own temperatures from raw logits, so they do not constrain the cache;
and a hard-label term, if present, always runs at T=1 independently. Freedom to choose, no
freedom to disagree.

Two live details are worth keeping. The coverage measurement is the striking one: the T=1
cache's top 64 held 99.5 percent of the mass, while the T=2 cache built from the *same
logits* held only about 70 percent, because dividing every logit by 2 flattens the softmax
enough to push nearly 30 percent of the probability past any fixed k. So a temperature
choice quietly moves the k knee from exercise 2 a long way: soften the teacher and you need
a much larger k for the same fidelity (that is also why the "matching" T=2 cached loss still
sat 0.21 below the dense T=2 KL; at this coverage, k=64 truncation is no longer a
few-percent effect). The manifest's temperature field is what lets you audit this after the
fact. And the failed spot check in step 3 is the whole argument for
manifests in one line: the tensors on disk were identical in the honest and dishonest runs;
only the recorded temperature let the protocol tell them apart.

**The paired trainings did not execute in this build; expected shape, not results:** the
T2/T2 arm should behave like Lab 03's T=2 soft arm (that is the point: it is the same
objective, cached), with the T2-vs-T1 comparison showing the usual soft-target trade
(slightly better ECE and agreement for T=2 on this family, per Lab 03's Part C). If you
instead run the mismatched T2-cache/T1-train arm deliberately, expect a loss that converges
to a nonzero floor and a student measurably *flatter* than the teacher, since it was trained
to imitate a softened target it believed was sharp.